# 14주 · 초기값 문제·Euler·RK2·RK4

수치해석및실습 CE273
원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 상미분방정식 (1) / 상미분방정식 (2)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week14/CE273_W14.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week14)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def ode_steps(f, start, y0, end, h, method="euler"):
    if not np.isfinite([start,y0,end,h]).all() or end<=start or h<=0:raise ValueError("유한한 초기조건·양의 구간·양의 h 필요")
    count=(end-start)/h;n=int(round(count))
    if not 1<=n<=100000 or not np.isclose(count,n,rtol=1e-12,atol=1e-12):raise ValueError("h로 구간을 정확히 분할할 양의 정수 스텝 수 필요")
    if method not in ["euler","heun","midpoint","ralston34","rk4"]:raise ValueError("방법 확인 필요")
    x=np.linspace(start,end,n+1);y=np.empty(n+1);y[0]=y0
    for i in range(n):
        k1=f(x[i],y[i])
        if method=="euler":slope=k1
        elif method=="heun":slope=(k1+f(x[i]+h,y[i]+h*k1))/2
        elif method=="midpoint":slope=f(x[i]+h/2,y[i]+h*k1/2)
        elif method=="ralston34":slope=k1/3+2*f(x[i]+3*h/4,y[i]+3*h*k1/4)/3
        else:
            k2=f(x[i]+h/2,y[i]+h*k1/2);k3=f(x[i]+h/2,y[i]+h*k2/2);k4=f(x[i]+h,y[i]+h*k3)
            slope=(k1+2*k2+2*k3+k4)/6
        y[i+1]=y[i]+h*slope
        if not np.isfinite(y[i+1]):raise ValueError("비유한 수치해: 스텝·모형·스케일 점검")
    return x,y


## S01. 원본 다항식 ODE와 오차

**슬라이드 연결**: CE273_W14a_ODE1.pptx p12, 13, 17, 19 / CE273_W14b_ODE2.pptx p10, 11, 12, 13, 14, 15, 16, 22

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 국소 한 스텝 오차와 누적 전역오차는 같은가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
f=lambda x,y:-2*x**3+12*x**2-20*x+8.5
truth=lambda x:-.5*x**4+4*x**3-10*x**2+8.5*x+1
for method in ["euler","midpoint","heun","ralston34","rk4"]:
    x,y=ode_steps(f,0,1,4,.5,method)
    print(method,"y(.5):",y[1],"y(4):",y[-1],"max_error:",max(abs(truth(x)-y)))
    if method=="midpoint":assert np.isclose(y[1],3.109375)
    if method=="ralston34":assert np.isclose(y[1],3.27734375)
    if method=="rk4":assert np.allclose(y,truth(x))
print("원본 Ralston RK2: c2=3/4, 가중치1/3·2/3. 다른 이름의 계수와 혼합 금지")
print("Ralston 첫 내부 y:",1+3/4*.5*8.5,"(원본 중간 표기 3.1875 정정)")

**조건 변경**: h=.5→.25 변경, 같은 종료점4에서 오차 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 원본 f·y0=1·구간0~4·h.5·명시적 방법·Ralston 계수·참해·국소/전역 구분 요청

**결과 검토**: 이 원본 f는 y에 독립. 내부 y 오타가 드러나지 않는 이유까지 검토

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. y 의존식의 Heun과 RK4 검산

**슬라이드 연결**: CE273_W14a_ODE1.pptx p24, 25, 26, 27

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 내부 y를 잘못 계산해도 모든 ODE에서 같은 결과가 나오는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
f=lambda x,y:4*np.exp(.8*x)-.5*y
truth=lambda x:4/1.3*(np.exp(.8*x)-np.exp(-.5*x))+2*np.exp(-.5*x)
for method in ["euler","heun","rk4"]:
    x,y=ode_steps(f,0,2,4,1.,method)
    print(method,"y1:",y[1],"y4:",y[-1],"error4:",abs(y[-1]-truth(4)))
    if method=="heun":assert np.isclose(y[1],6.701081856984936)
reference=integrate.solve_ivp(f,(0,4),[2.],rtol=1e-10,atol=1e-12)
assert np.isclose(reference.y[0,-1],truth(4),rtol=1e-9)
# 원본 14a의 반복 교정: 첫 구간에서 교정식을 반복
corrected=5.
for k in range(30):
    nxt=2+.5*(3+f(1,corrected))
    if abs(nxt-corrected)<1e-12:
        corrected=nxt;break
    corrected=nxt
print("Heun 반복 교정의 첫 구간:",corrected)
assert np.isclose(corrected,6.360865485587949)
print("Heun 반복 교정과 한 번 교정 RK2를 구분")

**조건 변경**: f가 y에도 의존하는 이번 예제에서 내부 단계의 영향 검토

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: f(x,y)·초기조건·구간·고정 스텝/적응 스텝·비교 오차 기준 명시

**결과 검토**: solve_ivp의 적응 스텝과 수동 RK의 고정 스텝을 같은 h라고 가정하지 않음

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

초기조건·단계값·오차 차수·독립 해석해 대조

입력 항목:

- `Euler_첫스텝`: 해당 계산값·항 수·오차 기준의 수치
- `Midpoint_첫스텝`: 해당 계산값·항 수·오차 기준의 수치
- `Ralston_원본변형_첫스텝`: 해당 계산값·항 수·오차 기준의 수치
- `Ralston_내부y_첫스텝`: 해당 계산값·항 수·오차 기준의 수치
- `전역오차와_국소오차는_같음`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'Euler_첫스텝': None, 'Midpoint_첫스텝': None, 'Ralston_원본변형_첫스텝': None, 'Ralston_내부y_첫스텝': None, '전역오차와_국소오차는_같음': None}


In [ ]:
def check_answers(values):
    expected = {'Euler_첫스텝': 5.25, 'Midpoint_첫스텝': 3.109375, 'Ralston_원본변형_첫스텝': 3.27734375, 'Ralston_내부y_첫스텝': 4.1875, '전역오차와_국소오차는_같음': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
